In [1]:
import gc
import warnings
from pathlib import Path
import optuna

import numpy as np
import pandas as pd

from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import balanced_accuracy_score
from sklearn.preprocessing import LabelEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline

import lightgbm as lgb
import xgboost as xgb
from catboost import CatBoostClassifier

warnings.filterwarnings("ignore")

/Users/apple/Documents Local/Mubashir Code/ML/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
SEED = 42
N_FOLDS = 100
USE_GPU = True
COMP_PATH = Path("/kaggle/input/competitions/playground-series-s6e4")
TARGET = "Irrigation_Need"
ID_COL = "id"
CLASS_ORDER = ["Low", "Medium", "High"]
TARGET_MAP = {"Low": 0, "Medium": 1, "High": 2}
REVERSE_TARGET_MAP = {0: "Low", 1: "Medium", 2: "High"}
np.random.seed(SEED)

In [3]:
def safe_div(a, b):
    return a / (b.replace(0, np.nan) + 1e-6)

In [5]:
df=pd.read_csv('train.csv')
print(len(df.columns))
df.columns


21


Index(['id', 'Soil_Type', 'Soil_pH', 'Soil_Moisture', 'Organic_Carbon',
       'Electrical_Conductivity', 'Temperature_C', 'Humidity', 'Rainfall_mm',
       'Sunlight_Hours', 'Wind_Speed_kmh', 'Crop_Type', 'Crop_Growth_Stage',
       'Season', 'Irrigation_Type', 'Water_Source', 'Field_Area_hectare',
       'Mulching_Used', 'Previous_Irrigation_mm', 'Region', 'Irrigation_Need'],
      dtype='str')

In [6]:
df.isnull().sum()

id                         0
Soil_Type                  0
Soil_pH                    0
Soil_Moisture              0
Organic_Carbon             0
Electrical_Conductivity    0
Temperature_C              0
Humidity                   0
Rainfall_mm                0
Sunlight_Hours             0
Wind_Speed_kmh             0
Crop_Type                  0
Crop_Growth_Stage          0
Season                     0
Irrigation_Type            0
Water_Source               0
Field_Area_hectare         0
Mulching_Used              0
Previous_Irrigation_mm     0
Region                     0
Irrigation_Need            0
dtype: int64

In [8]:
df_t=pd.read_csv('test.csv')
df_t.isnull().sum()

id                         0
Soil_Type                  0
Soil_pH                    0
Soil_Moisture              0
Organic_Carbon             0
Electrical_Conductivity    0
Temperature_C              0
Humidity                   0
Rainfall_mm                0
Sunlight_Hours             0
Wind_Speed_kmh             0
Crop_Type                  0
Crop_Growth_Stage          0
Season                     0
Irrigation_Type            0
Water_Source               0
Field_Area_hectare         0
Mulching_Used              0
Previous_Irrigation_mm     0
Region                     0
dtype: int64

In [9]:
cat_cols=df.select_dtypes(include='object').columns.tolist()
print(cat_cols)
print(len(cat_cols))

['Soil_Type', 'Crop_Type', 'Crop_Growth_Stage', 'Season', 'Irrigation_Type', 'Water_Source', 'Mulching_Used', 'Region', 'Irrigation_Need']
9


In [10]:
cat_cols=df.select_dtypes(include='number').columns.tolist()
print(cat_cols)
print(len(cat_cols))

['id', 'Soil_pH', 'Soil_Moisture', 'Organic_Carbon', 'Electrical_Conductivity', 'Temperature_C', 'Humidity', 'Rainfall_mm', 'Sunlight_Hours', 'Wind_Speed_kmh', 'Field_Area_hectare', 'Previous_Irrigation_mm']
12


In [11]:
def add_basic_features(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()

    expected_cols = ['Soil_Type', 'Soil_pH', 'Soil_Moisture', 'Organic_Carbon','Electrical_Conductivity','Temperature_C',
                     'Humidity', 'Rainfall_mm','Sunlight_Hours', 'Wind_Speed_kmh','Crop_Type', 'Crop_Growth_Stage','Season', 
                     'Irrigation_Type', 'Water_Source', 'Field_Area_hectare','Mulching_Used', 'Previous_Irrigation_mm', 'Region']

    # standardize categorical feature
    cat_cols = df.select_dtypes(include='object').columns.tolist()
    for c in cat_cols:
        df[c] = df[c].astype(str).fillna("Missing")

    # numerical cleanup
    num_cols = df.select_dtypes(include='number').columns.tolist()
    for c in num_cols:
        df[c] = pd.to_numeric(df[c], errors="coerce")
        
    # Domain-inspired features
    df["Total_Water_Input"] = df["Rainfall_mm"] + df["Previous_Irrigation_mm"]
    df["Temp_Humidity_Gap"] = df["Temperature_C"] - df["Humidity"]
    df["Evaporative_Stress"] = (df["Temperature_C"]* df["Sunlight_Hours"]* (df["Wind_Speed_kmh"] + 1) ) / (df["Humidity"] + 10)
    df["Water_Deficit"] = df["Total_Water_Input"] - df["Evaporative_Stress"]
    df["Moisture_Temp"] = df["Soil_Moisture"] * df["Temperature_C"]
    df["Moisture_Humidity"] = df["Soil_Moisture"] * df["Humidity"]
    df["Rain_x_Moisture"] = df["Rainfall_mm"] * df["Soil_Moisture"]
    df["Irrigation_x_Area"] = df["Previous_Irrigation_mm"] * df["Field_Area_hectare"]
    df["Sun_Wind"] = df["Sunlight_Hours"] * df["Wind_Speed_kmh"]
    df["Soil_Health_Index"] = (df["Organic_Carbon"] * (df["Soil_Moisture"] + 1)) / (df["Electrical_Conductivity"] + 1)
    df["pH_Deviation_65"] = (df["Soil_pH"].fillna(6.5) - 6.5).abs()
    df["Water_per_Area"] = safe_div(df["Total_Water_Input"], df["Field_Area_hectare"] + 1)
    df["Rain_per_Area"] = safe_div(df["Rainfall_mm"], df["Field_Area_hectare"] + 1)
    df["PrevIrr_per_Area"] = safe_div(df["Previous_Irrigation_mm"], df["Field_Area_hectare"] + 1)
    df["Dryness_Index"] = (df["Temperature_C"] * (100 - df["Humidity"]).clip(lower=0)) / (df["Rainfall_mm"] + 1)

    # bins converted to string so tree models can also use them as categorical
    df["Moisture_Bin"] = pd.cut(df["Soil_Moisture"], bins=6, labels=False, duplicates="drop").astype("float")
    df["Rainfall_Bin"] = pd.cut(df["Rainfall_mm"], bins=6, labels=False, duplicates="drop").astype("float")
    df["Temp_Bin"] = pd.cut(df["Temperature_C"], bins=6, labels=False, duplicates="drop").astype("float")

    # crossed categoricals
    df["Crop_Season"] = df["Crop_Type"].astype(str) + "__" + df["Season"].astype(str)
    df["Crop_Stage"] = df["Crop_Type"].astype(str) + "__" + df["Crop_Growth_Stage"].astype(str)
    df["Soil_Region"] = df["Soil_Type"].astype(str) + "__" + df["Region"].astype(str)
    df["Water_Method_Source"] = df["Irrigation_Type"].astype(str) + "__" + df["Water_Source"].astype(str)

    # missing counts
    df["row_missing_count"] = df.isna().sum(axis=1)
    return df


def frequency_encode(train_df, test_df, cols):
    train_df = train_df.copy()
    test_df = test_df.copy()
    for col in cols:
        freq = pd.concat([train_df[col], test_df[col]], axis=0).astype(str).value_counts(dropna=False)
        train_df[f"{col}_freq"] = train_df[col].astype(str).map(freq).astype(float)
        test_df[f"{col}_freq"] = test_df[col].astype(str).map(freq).astype(float)
    return train_df, test_df

In [13]:
train_df=pd.read_csv('train.csv') 
test_df =pd.read_csv('test.csv')

train_df = add_basic_features(train_df)
test_df = add_basic_features(test_df)

In [14]:
len(train_df.columns)

44

In [15]:
cat_cl=train_df.select_dtypes(include='object').columns.tolist()
print(cat_cl)
print(len(cat_cl))

['Soil_Type', 'Crop_Type', 'Crop_Growth_Stage', 'Season', 'Irrigation_Type', 'Water_Source', 'Mulching_Used', 'Region', 'Irrigation_Need', 'Crop_Season', 'Crop_Stage', 'Soil_Region', 'Water_Method_Source']
13


In [16]:
base_cat_cols = train_df.select_dtypes(include='object').columns.tolist()
base_cat_cols.remove(TARGET)
train_df, test_df = frequency_encode(train_df, test_df, base_cat_cols)

# unified category space for categorical models
for col in base_cat_cols:
    all_vals = pd.concat([train_df[col], test_df[col]], axis=0).astype(str).fillna("Missing").unique()
    train_df[col] = pd.Categorical(train_df[col].astype(str).fillna("Missing"), categories=all_vals)
    test_df[col] = pd.Categorical(test_df[col].astype(str).fillna("Missing"), categories=all_vals)

# Final feature list
ignore_cols = [ID_COL, TARGET]
feature_cols = [c for c in train_df.columns if c not in ignore_cols]
cat_cols = [c for c in feature_cols if str(train_df[c].dtype) == "category"]
num_cols = [c for c in feature_cols if c not in cat_cols]

print(f"Total features: {len(feature_cols)}")
print(f"Categorical:    {len(cat_cols)}")
print(f"Numerical:      {len(num_cols)}")

X = train_df[feature_cols].copy()
X_test = test_df[feature_cols].copy()
y = train_df[TARGET].map(TARGET_MAP).to_numpy()
test_ids = test_df[ID_COL].to_numpy()



Total features: 54
Categorical:    12
Numerical:      42


# model parameter config

In [23]:
def get_lgb_params():
    params = {
        "objective": "multiclass",
        "num_class": 3,
        "metric": "multi_logloss",
        "boosting_type": "gbdt",
        "learning_rate": 0.1,
        "n_estimators": 510,
        "num_leaves": 127,
        "max_depth": -1,
        "min_child_samples": 25,
        "subsample": 0.85,
        "subsample_freq": 1,
        "colsample_bytree": 0.80,
        "reg_alpha": 0.2,
        "reg_lambda": 2.0,
        "random_state": SEED,
        "verbose": -1,
    }
    return params

def get_xgb_params():
    params = {
        "objective": "multi:softprob",
        "num_class": 3,
        "eval_metric": "mlogloss",
        "n_estimators": 500,
        "learning_rate": 0.1,
        "max_depth": 8,
        "min_child_weight": 2,
        "subsample": 0.85,
        "colsample_bytree": 0.80,
        "reg_alpha": 0.1,
        "reg_lambda": 2.0,
        "random_state": SEED,
        "enable_categorical": True,
    }
    if USE_GPU:
        params["tree_method"] = "hist"

        params["device"] = "cuda"
    else:
        params["tree_method"] = "hist"
    return params


def get_cat_params():
    params = {
        "loss_function": "MultiClass",
        "eval_metric": "MultiClass",
        "iterations": 700,
        "learning_rate": 0.1,
        "depth": 8,
        "l2_leaf_reg": 5.0,
        "random_seed": SEED,
        "auto_class_weights": "Balanced",
        "verbose": 0,
        "allow_writing_files": False,
    }
    if USE_GPU:
        params["task_type"] = "GPU"
    return params



# CV folds

In [24]:
skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)

oof_lgb = np.zeros((len(X), 3), dtype=np.float32)
oof_xgb = np.zeros((len(X), 3), dtype=np.float32)
oof_cat = np.zeros((len(X), 3), dtype=np.float32)

test_lgb = np.zeros((len(X_test), 3), dtype=np.float32)
test_xgb = np.zeros((len(X_test), 3), dtype=np.float32)
test_cat = np.zeros((len(X_test), 3), dtype=np.float32)

fold_scores = []

In [26]:
for fold, (tr_idx, va_idx) in enumerate(skf.split(X, y), 1):
    print("=" * 70)
    print(f"Fold {fold}/{N_FOLDS}")
    print("=" * 70)

    X_tr = X.iloc[tr_idx].copy()
    X_va = X.iloc[va_idx].copy()
    y_tr = y[tr_idx]
    y_va = y[va_idx]

    # LightGBM 
    lgb_model = lgb.LGBMClassifier(**get_lgb_params())
    lgb_model.fit(
        X_tr,
        y_tr,
        eval_set=[(X_va, y_va)],
        eval_metric="multi_logloss",
        categorical_feature=cat_cols,
        callbacks=[lgb.early_stopping(200, verbose=False)],
    )
    oof_lgb[va_idx] = lgb_model.predict_proba(X_va)
    test_lgb += lgb_model.predict_proba(X_test) / N_FOLDS
    lgb_score = balanced_accuracy_score(y_va, oof_lgb[va_idx].argmax(axis=1))
    print(f"LightGBM fold balanced accuracy: {lgb_score:.6f}")

    #  XGBoost 
    xgb_model = xgb.XGBClassifier(**get_xgb_params())
    xgb_model.fit(
        X_tr,
        y_tr,
        eval_set=[(X_va, y_va)],
        verbose=False,
    )
    oof_xgb[va_idx] = xgb_model.predict_proba(X_va)

    test_xgb += xgb_model.predict_proba(X_test) / N_FOLDS
    xgb_score = balanced_accuracy_score(y_va, oof_xgb[va_idx].argmax(axis=1))
    print(f"XGBoost  fold balanced accuracy: {xgb_score:.6f}")

    # CatBoost
    cat_model = CatBoostClassifier(**get_cat_params(), cat_features=cat_cols)
    cat_model.fit(
        X_tr,
        y_tr,
        eval_set=(X_va, y_va),
        use_best_model=True,
    )
    oof_cat[va_idx] = cat_model.predict_proba(X_va)
    test_cat += cat_model.predict_proba(X_test) / N_FOLDS
    cat_score = balanced_accuracy_score(y_va, oof_cat[va_idx].argmax(axis=1))
    print(f"CatBoost  fold balanced accuracy: {cat_score:.6f}")

    # fast weighted average snapshot for monitoring
    avg_fold = 0.30 * oof_lgb[va_idx] + 0.25 * oof_xgb[va_idx] + 0.45 * oof_cat[va_idx]
    avg_fold_score = balanced_accuracy_score(y_va, avg_fold.argmax(axis=1))
    fold_scores.append(avg_fold_score)
    print(f"Weighted avg fold balanced accuracy: {avg_fold_score:.6f}")

    del X_tr, X_va, y_tr, y_va, lgb_model, xgb_model, cat_model
    gc.collect()


print("\nOOF base scores")
print(f"LightGBM : {balanced_accuracy_score(y, oof_lgb.argmax(axis=1)):.6f}")
print(f"XGBoost  : {balanced_accuracy_score(y, oof_xgb.argmax(axis=1)):.6f}")
print(f"CatBoost : {balanced_accuracy_score(y, oof_cat.argmax(axis=1)):.6f}")
print(f"Avg folds : {np.mean(fold_scores):.6f}")



Fold 1/100


KeyboardInterrupt: 

In [ ]:
test_prob = ( test_lgb +  test_xgb +  test_cat)/3

final_test_pre = test_prob.argmax(axis=1)
final_test_labels1 = [REVERSE_TARGET_MAP[int(x)] for x in final_test_pre]

submission = pd.DataFrame({
    ID_COL: test_ids,
    TARGET: final_test_labels1,
})
submission.to_csv("submission_bse_model.csv", index=False)

print("\nSubmission saved: submission_bse_model.csv")
print(submission.head())
print("\nPrediction distribution:")
print(submission[TARGET].value_counts(normalize=True))



Submission saved: submission_bse_model.csv
       id Irrigation_Need
0  630000             Low
1  630001             Low
2  630002             Low
3  630003             Low
4  630004             Low

Prediction distribution:
Irrigation_Need
Low       0.592030
Medium    0.375593
High      0.032378
Name: proportion, dtype: float64


In [ ]:
test_prob = ( test_lgb +  test_cat)/2

final_test_pre = test_prob.argmax(axis=1)
final_test_labels1 = [REVERSE_TARGET_MAP[int(x)] for x in final_test_pre]

submission = pd.DataFrame({
    ID_COL: test_ids,
    TARGET: final_test_labels1,
})
submission.to_csv("submission_xgb_cat.csv", index=False)

print("\nSubmission saved: submission_xgb_cat.csv")
print(submission.head())
print("\nPrediction distribution:")
print(submission[TARGET].value_counts(normalize=True))



Submission saved: submission_xgb_cat.csv
       id Irrigation_Need
0  630000             Low
1  630001             Low
2  630002             Low
3  630003             Low
4  630004             Low

Prediction distribution:
Irrigation_Need
Low       0.591878
Medium    0.375215
High      0.032907
Name: proportion, dtype: float64


# BLEND SEARCH


In [ ]:
print("\nSearching good blend weights...")
best_blend_score = -1
best_blend = None
best_oof_probs = None
best_test_probs = None

for wl in np.arange(0.10, 0.51, 0.05):
    for wx in np.arange(0.10, 0.41, 0.05):
        wc = 1.0 - wl - wx
        if wc < 0.20 or wc > 0.70:
            continue
        oof_blend = wl * oof_lgb + wx * oof_xgb + wc * oof_cat
        score = balanced_accuracy_score(y, oof_blend.argmax(axis=1))
        if score > best_blend_score:
            best_blend_score = score
            best_blend = (round(wl, 4), round(wx, 4), round(wc, 4))
            best_oof_probs = oof_blend.copy()
            best_test_probs = wl * test_lgb + wx * test_xgb + wc * test_cat

print(f"Best manual blend (lgb, xgb, cat): {best_blend}")

print(f"Best manual blend OOF balanced accuracy: {best_blend_score:.6f}")


Searching good blend weights...
Best manual blend (lgb, xgb, cat): (np.float64(0.15), np.float64(0.15), np.float64(0.7))
Best manual blend OOF balanced accuracy: 0.966600


In [ ]:
final_test_pred1 = best_test_probs.argmax(axis=1)
final_test_labels1 = [REVERSE_TARGET_MAP[int(x)] for x in final_test_pred1]

submission = pd.DataFrame({
    ID_COL: test_ids,
    TARGET: final_test_labels1,
})
submission.to_csv("submission_best_blend.csv", index=False)

print("\nSubmission saved: submission_best_blend.csv")
print(submission.head())
print("\nPrediction distribution:")
print(submission[TARGET].value_counts(normalize=True))



Submission saved: submission_best_blend.csv
       id Irrigation_Need
0  630000             Low
1  630001             Low
2  630002             Low
3  630003             Low
4  630004             Low

Prediction distribution:
Irrigation_Need
Low       0.591593
Medium    0.374752
High      0.033656
Name: proportion, dtype: float64


In [ ]:
print("\nTraining stacker...")
stack_train = np.hstack([oof_lgb, oof_xgb, oof_cat])
stack_test = np.hstack([test_lgb, test_xgb, test_cat])

meta_oof = np.zeros((len(X), 3), dtype=np.float32)
meta_test = np.zeros((len(X_test), 3), dtype=np.float32)

for fold, (tr_idx, va_idx) in enumerate(skf.split(stack_train, y), 1):
    meta = Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("clf", LogisticRegression(max_iter=4000, class_weight="balanced", C=0.5, random_state=SEED)),
    ])
    meta.fit(stack_train[tr_idx], y[tr_idx])
    meta_oof[va_idx] = meta.predict_proba(stack_train[va_idx])
    meta_test += meta.predict_proba(stack_test) / N_FOLDS

stack_score = balanced_accuracy_score(y, meta_oof.argmax(axis=1))
print(f"Stacker OOF balanced accuracy: {stack_score:.6f}")

if stack_score >= best_blend_score:
    print("Using stacker outputs.")
    final_oof_probs = meta_oof.copy()
    final_test_probs = meta_test.copy()
else:
    print("Using manual blend outputs.")
    final_oof_probs = best_oof_probs.copy()
    final_test_probs = best_test_probs.copy()

raw_final_score = balanced_accuracy_score(y, final_oof_probs.argmax(axis=1))
print(f"Raw final OOF balanced accuracy: {raw_final_score:.6f}")


Training stacker...
Stacker OOF balanced accuracy: 0.969721
Using stacker outputs.
Raw final OOF balanced accuracy: 0.969721


In [ ]:
final_test_pred2 = final_test_probs.argmax(axis=1)
final_test_labels2 = [REVERSE_TARGET_MAP[int(x)] for x in final_test_pred2]

submission = pd.DataFrame({
    ID_COL: test_ids,
    TARGET: final_test_labels2,
})
submission.to_csv("submission_meta.csv", index=False)

print("\nSubmission saved: submission_raw.csv")
print(submission.head())
print("\nPrediction distribution:")
print(submission[TARGET].value_counts(normalize=True))



Submission saved: submission_raw.csv
       id Irrigation_Need
0  630000             Low
1  630001             Low
2  630002             Low
3  630003             Low
4  630004             Low

Prediction distribution:
Irrigation_Need
Low       0.591637
Medium    0.370700
High      0.037663
Name: proportion, dtype: float64


In [ ]:
print("\nTuning class weights for balanced accuracy...")
best_class_weights = np.array([1.0, 1.0, 1.0], dtype=np.float32)
best_tuned_score = raw_final_score

for w_low in np.arange(0.85, 1.21, 0.05):
    for w_med in np.arange(0.85, 1.61, 0.05):
        for w_high in np.arange(0.85, 2.51, 0.05):
            w = np.array([w_low, w_med, w_high], dtype=np.float32)
            preds = (final_oof_probs * w).argmax(axis=1)
            score = balanced_accuracy_score(y, preds)
            if score > best_tuned_score:
                best_tuned_score = score
                best_class_weights = w.copy()

print(f"Best class weights: {best_class_weights}")
print(f"Tuned OOF balanced accuracy: {best_tuned_score:.6f}")
print(f"Gain over raw final: {best_tuned_score - raw_final_score:.6f}")



Tuning class weights for balanced accuracy...
Best class weights: [1.1  1.1  1.55]
Tuned OOF balanced accuracy: 0.969808
Gain over raw final: 0.000087


In [ ]:
final_test_pred = (final_test_probs * best_class_weights).argmax(axis=1)
final_test_labels = [REVERSE_TARGET_MAP[int(x)] for x in final_test_pred]

submission = pd.DataFrame({
    ID_COL: test_ids,
    TARGET: final_test_labels,
})
submission.to_csv("submission.csv", index=False)

print("\nSubmission saved: submission.csv")
print(submission.head())
print("\nPrediction distribution:")
print(submission[TARGET].value_counts(normalize=True))



Submission saved: submission.csv
       id Irrigation_Need
0  630000             Low
1  630001             Low
2  630002             Low
3  630003             Low
4  630004             Low

Prediction distribution:
Irrigation_Need
Low       0.591637
Medium    0.369430
High      0.038933
Name: proportion, dtype: float64


In [ ]:
final_oof_pred = (final_oof_probs * best_class_weights).argmax(axis=1)
print("\nFinal tuned OOF balanced accuracy:", balanced_accuracy_score(y, final_oof_pred))
print("Class distribution in train:", pd.Series(y).map(REVERSE_TARGET_MAP).value_counts(normalize=True).sort_index().to_dict())
print("Done.")


Final tuned OOF balanced accuracy: 0.9698080643547251
Class distribution in train: {'High': 0.03334761904761905, 'Low': 0.5871698412698413, 'Medium': 0.3794825396825397}
Done.


In [ ]:
print("\nTuning class weights for balanced accuracy using Optuna...")

# Define objective function for Optuna
def objective(trial):
    # Suggest a weight for each class (continuous range)
    w_low = trial.suggest_float("w_low", 0.85, 1.2)
    w_med = trial.suggest_float("w_med", 0.85, 1.6)
    w_high = trial.suggest_float("w_high", 0.85, 2.5)

    w = np.array([w_low, w_med, w_high], dtype=np.float32)
    # Apply weights to OOF probabilities and get predicted classes
    preds = (final_oof_probs * w).argmax(axis=1)
    # Return negative because Optuna minimizes by default, but we want to maximize score
    return -balanced_accuracy_score(y, preds)

# Create study and optimize
study = optuna.create_study(direction="minimize")  # minimizing negative score = maximizing score
study.optimize(objective, n_trials=450, show_progress_bar=True)

# Extract best weights
best_class_weights = np.array([
    study.best_params["w_low"],
    study.best_params["w_med"],
    study.best_params["w_high"]
], dtype=np.float32)

# Apply best weights to OOF to get best tuned score
best_tuned_score = balanced_accuracy_score(y, (final_oof_probs * best_class_weights).argmax(axis=1))

print(f"Best class weights from Optuna: {best_class_weights}")
print(f"Tuned OOF balanced accuracy: {best_tuned_score:.6f}")
print(f"Gain over raw final: {best_tuned_score - raw_final_score:.6f}")

# Apply best weights to test predictions
final_test_pred = (final_test_probs * best_class_weights).argmax(axis=1)
final_test_labels = [REVERSE_TARGET_MAP[int(x)] for x in final_test_pred]

# Prepare submission
submission = pd.DataFrame({
    ID_COL: test_ids,
    TARGET: final_test_labels,
})
submission.to_csv("submission_optuna.csv", index=False)

print("\nSubmission saved: submission.csv")
print(submission.head())
print("\nPrediction distribution:")
print(submission[TARGET].value_counts(normalize=True))

[I 2026-04-08 12:34:06,673] A new study created in memory with name: no-name-51c546a0-54d3-49d7-8f62-c89f9e40d628



Tuning class weights for balanced accuracy using Optuna...


  0%|          | 0/450 [00:00<?, ?it/s]

[I 2026-04-08 12:34:06,708] Trial 0 finished with value: -0.9696231163709479 and parameters: {'w_low': 0.9052569827092072, 'w_med': 1.1348118584493325, 'w_high': 2.375070830493082}. Best is trial 0 with value: -0.9696231163709479.
[I 2026-04-08 12:34:06,731] Trial 1 finished with value: -0.9696374441656935 and parameters: {'w_low': 1.164705341397583, 'w_med': 1.560779050474751, 'w_high': 1.9324746327494835}. Best is trial 1 with value: -0.9696374441656935.
[I 2026-04-08 12:34:06,754] Trial 2 finished with value: -0.9696314898831746 and parameters: {'w_low': 0.8551984531734815, 'w_med': 1.5641467610528976, 'w_high': 1.0585730462252994}. Best is trial 1 with value: -0.9696374441656935.
[I 2026-04-08 12:34:06,779] Trial 3 finished with value: -0.9695901671704363 and parameters: {'w_low': 0.8832071611432598, 'w_med': 1.1578445974794014, 'w_high': 2.089407276795906}. Best is trial 1 with value: -0.9696374441656935.
[I 2026-04-08 12:34:06,805] Trial 4 finished with value: -0.9697440827357293